<a href="https://colab.research.google.com/github/thapliyalbani/OS-Assignment/blob/main/os%20assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Program 1: Introduction to Linux System Calls, Process Creation using fork(), exec() and wait()

In [1]:
import os

print("Parent started, PID =", os.getpid())

pid = os.fork()

if pid == 0:
    print("\nChild: PID =", os.getpid(), ", Parent PID =", os.getppid())
    print("Child: now running 'date' using exec()")
    os.execvp("date", ["date"])
    print("This line never runs")
else:

    print("Parent: created child with PID =", pid)
    child_pid, status = os.wait()
    print("\nParent: child", child_pid, "finished")
    print("Parent: exit status =", os.WEXITSTATUS(status))
    print("Parent: done")

Parent started, PID = 1073

Child: PID = 1094 , Parent PID = 1073
Child: now running 'date' using exec()
Parent started, PID = 1073
Parent: created child with PID = 1094

Parent: child 1094 finished
Parent: exit status = 0
Parent: done


/tmp/ipykernel_1073/1304034181.py:5: DeprecationWarning: This process (pid=1073) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


Program 2: Thread Creation and Management using POSIX Threads (pthreads)

In [2]:
import threading
import time

results = {}

def worker(thread_id):
    print(f"Thread {thread_id}: started (name = {threading.current_thread().name})")
    time.sleep(1)                  # simulate some work
    print(f"Thread {thread_id}: finished")
    results[thread_id] = thread_id * 10   # store result for main thread

NUM_THREADS = 3
threads = []

print(f"Main: creating {NUM_THREADS} threads")

for i in range(1, NUM_THREADS + 1):
    t = threading.Thread(target=worker, args=(i,), name=f"Worker-{i}")
    threads.append(t)
    t.start()

for t in threads:
    t.join()

for tid, value in results.items():
    print(f"Main: thread {tid} returned {value}")

print("Main: all threads completed")


Main: creating 3 threads
Thread 1: started (name = Worker-1)
Thread 2: started (name = Worker-2)
Thread 3: started (name = Worker-3)
Thread 1: finishedThread 2: finished
Thread 3: finished

Main: thread 2 returned 20
Main: thread 3 returned 30
Main: thread 1 returned 10
Main: all threads completed


Program 3: CPU Scheduling Algorithms — FCFS, SJF, Priority and Round Robin

In [3]:
MAX = 20
def printResult(p):
    n = len(p)
    totalWT = 0
    totalTAT = 0

    print("\nPID\tAT\tBT\tCT\tTAT\tWT")

    for process in p:
        print(
            f"{process['pid']}\t"
            f"{process['at']}\t"
            f"{process['bt']}\t"
            f"{process['ct']}\t"
            f"{process['tat']}\t"
            f"{process['wt']}"
        )

        totalWT += process['wt']
        totalTAT += process['tat']

    print(f"Average Waiting Time    = {totalWT / n:.2f}")
    print(f"Average Turnaround Time = {totalTAT / n:.2f}")

# 1. First Come First Serve (FCFS)
def fcfs(p):
    time = 0
    for process in p:
        if time < process['at']:
            time = process['at']

        time += process['bt']

        process['ct'] = time
        process['tat'] = process['ct'] - process['at']
        process['wt'] = process['tat'] - process['bt']

    printResult(p)

# 2. Shortest Job First (SJF) - Non-preemptive
def sjf(p):
    n = len(p)
    completed = 0
    time = 0
    done = [False] * n

    while completed < n:
        idx = -1
        minBT = float('inf')

        for i in range(n):
            if (not done[i] and
                    p[i]['at'] <= time and
                    p[i]['bt'] < minBT):

                minBT = p[i]['bt']
                idx = i

        if idx == -1:
            time += 1
            continue

        time += p[idx]['bt']

        p[idx]['ct'] = time
        p[idx]['tat'] = p[idx]['ct'] - p[idx]['at']
        p[idx]['wt'] = p[idx]['tat'] - p[idx]['bt']

        done[idx] = True
        completed += 1

    printResult(p)


# 3. Priority Scheduling - Non-preemptive
# Smaller priority number means higher priority
def priorityScheduling(p):
    n = len(p)
    completed = 0
    time = 0
    done = [False] * n

    while completed < n:
        idx = -1
        best = float('inf')

        for i in range(n):
            if (not done[i] and
                    p[i]['at'] <= time and
                    p[i]['pr'] < best):

                best = p[i]['pr']
                idx = i

        if idx == -1:
            time += 1
            continue

        time += p[idx]['bt']

        p[idx]['ct'] = time
        p[idx]['tat'] = p[idx]['ct'] - p[idx]['at']
        p[idx]['wt'] = p[idx]['tat'] - p[idx]['bt']

        done[idx] = True
        completed += 1

    printResult(p)


# 4. Round Robin Scheduling
def roundRobin(p, quantum):
    n = len(p)
    time = 0
    completed = 0

    for process in p:
        process['rem'] = process['bt']

    queue = []
    inQueue = [False] * n

    # Add processes arriving at time 0
    for i in range(n):
        if p[i]['at'] <= time:
            queue.append(i)
            inQueue[i] = True

    front = 0

    while completed < n:

        # If queue is empty, advance time
        if front == len(queue):
            time += 1

            for i in range(n):
                if (not inQueue[i] and
                        p[i]['rem'] > 0 and
                        p[i]['at'] <= time):

                    queue.append(i)
                    inQueue[i] = True

            continue

        idx = queue[front]
        front += 1

        slice_time = min(p[idx]['rem'], quantum)

        time += slice_time
        p[idx]['rem'] -= slice_time

        # Add newly arrived processes to the queue
        for i in range(n):
            if (not inQueue[i] and
                    p[i]['rem'] > 0 and
                    p[i]['at'] <= time and
                    i != idx):

                queue.append(i)
                inQueue[i] = True

        if p[idx]['rem'] > 0:
            queue.append(idx)

        else:
            p[idx]['ct'] = time
            p[idx]['tat'] = p[idx]['ct'] - p[idx]['at']
            p[idx]['wt'] = p[idx]['tat'] - p[idx]['bt']

            completed += 1
            inQueue[idx] = False

    printResult(p)


# Main Program
if __name__ == "__main__":

    n = int(input("Enter number of processes: "))

    processes = []

    for i in range(n):
        print(f"\nEnter details for Process {i + 1}")

        pid = int(input("Enter Process ID: "))
        at = int(input("Enter Arrival Time: "))
        bt = int(input("Enter Burst Time: "))
        pr = int(input("Enter Priority: "))

        processes.append({
            'pid': pid,
            'at': at,
            'bt': bt,
            'pr': pr,
            'ct': 0,
            'tat': 0,
            'wt': 0,
            'rem': 0
        })

    print("\nChoose Scheduling Algorithm:")
    print("1. FCFS")
    print("2. SJF")
    print("3. Priority Scheduling")
    print("4. Round Robin")

    choice = int(input("Enter your choice: "))

    if choice == 1:
        fcfs([process.copy() for process in processes])

    elif choice == 2:
        sjf([process.copy() for process in processes])

    elif choice == 3:
        priorityScheduling([process.copy() for process in processes])

    elif choice == 4:
        quantum = int(input("Enter Time Quantum: "))

        if quantum <= 0:
            print("Time Quantum must be greater than zero.")
        else:
            roundRobin(
                [process.copy() for process in processes],
                quantum
            )

    else:
        print("Invalid choice!")



Enter number of processes: 4

Enter details for Process 1
Enter Process ID: 1
Enter Arrival Time: 0
Enter Burst Time: 5
Enter Priority: 2

Enter details for Process 2
Enter Process ID: 2
Enter Arrival Time: 1
Enter Burst Time: 3
Enter Priority: 1

Enter details for Process 3
Enter Process ID: 3
Enter Arrival Time: 2
Enter Burst Time: 8
Enter Priority: 3

Enter details for Process 4
Enter Process ID: 4
Enter Arrival Time: 4
Enter Burst Time: 3
Enter Priority: 4

Choose Scheduling Algorithm:
1. FCFS
2. SJF
3. Priority Scheduling
4. Round Robin
Enter your choice: 1

PID	AT	BT	CT	TAT	WT
1	0	5	5	5	0
2	1	3	8	7	4
3	2	8	16	14	6
4	4	3	19	15	12
Average Waiting Time    = 5.50
Average Turnaround Time = 10.25


Program 4 Process Synchronization — Producer-Consumer Problem using Semaphores

In [4]:
import threading
import time

BUFFER_SIZE = 5
ITEMS = 10

buffer = [0] * BUFFER_SIZE
in_index = 0
out_index = 0

# Semaphores
empty = threading.Semaphore(BUFFER_SIZE)
full = threading.Semaphore(0)
mutex = threading.Semaphore(1)


def producer():
    global in_index

    for i in range(1, ITEMS + 1):
        empty.acquire()
        mutex.acquire()

        buffer[in_index] = i
        print(f"Producer produced item {i} at slot {in_index}")

        in_index = (in_index + 1) % BUFFER_SIZE

        mutex.release()
        full.release()

        time.sleep(0.1)


def consumer():
    global out_index

    for i in range(1, ITEMS + 1):
        full.acquire()
        mutex.acquire()

        item = buffer[out_index]
        print(
            f"\t\tConsumer consumed item {item} "
            f"from slot {out_index}"
        )

        out_index = (out_index + 1) % BUFFER_SIZE

        mutex.release()
        empty.release()

        time.sleep(0.15)


def main():
    # Create producer and consumer threads
    prod = threading.Thread(target=producer)
    cons = threading.Thread(target=consumer)

    # Start threads
    prod.start()
    cons.start()

    # Wait for both threads to finish
    prod.join()
    cons.join()

    print("All items produced and consumed.")


if __name__ == "__main__":
    main()


Producer produced item 1 at slot 0
		Consumer consumed item 1 from slot 0
Producer produced item 2 at slot 1
		Consumer consumed item 2 from slot 1
Producer produced item 3 at slot 2
Producer produced item 4 at slot 3
		Consumer consumed item 3 from slot 2
Producer produced item 5 at slot 4
		Consumer consumed item 4 from slot 3
Producer produced item 6 at slot 0
		Consumer consumed item 5 from slot 4
Producer produced item 7 at slot 1
Producer produced item 8 at slot 2
		Consumer consumed item 6 from slot 0
Producer produced item 9 at slot 3
		Consumer consumed item 7 from slot 1
Producer produced item 10 at slot 4
		Consumer consumed item 8 from slot 2
		Consumer consumed item 9 from slot 3
		Consumer consumed item 10 from slot 4
All items produced and consumed.


Program 5: Process Synchronization — Readers-Writers Problem using Semaphores

In [5]:
import threading
import time

# Semaphores
mutex = threading.Semaphore(1)
wrt = threading.Semaphore(1)

read_count = 0
shared_data = 0


def reader(id):
    global read_count

    # Entry section
    mutex.acquire()

    read_count += 1

    if read_count == 1:
        wrt.acquire()  # First reader locks the resource

    mutex.release()

    # Reading section
    print(f"Reader {id}: read shared_data = {shared_data}")

    time.sleep(0.2)

    # Exit section
    mutex.acquire()

    read_count -= 1

    if read_count == 0:
        wrt.release()  # Last reader unlocks the resource

    mutex.release()


def writer(id):
    global shared_data

    # Acquire exclusive access
    wrt.acquire()

    shared_data += 1

    print(f"Writer {id}: wrote shared_data = {shared_data}")

    time.sleep(0.3)

    wrt.release()


def main():
    readers = []
    writers = []

    # Create the first writer
    w1 = threading.Thread(target=writer, args=(1,))
    writers.append(w1)
    w1.start()

    # Create five readers
    for i in range(1, 6):
        r = threading.Thread(target=reader, args=(i,))
        readers.append(r)
        r.start()

    # Create the second writer
    w2 = threading.Thread(target=writer, args=(2,))
    writers.append(w2)
    w2.start()

    # Wait for all readers to finish
    for r in readers:
        r.join()

    # Wait for all writers to finish
    for w in writers:
        w.join()

    print("All readers and writers have finished.")


if __name__ == "__main__":
    main()



Writer 1: wrote shared_data = 1
Reader 1: read shared_data = 1
Reader 2: read shared_data = 1
Reader 3: read shared_data = 1
Reader 4: read shared_data = 1
Reader 5: read shared_data = 1
Writer 2: wrote shared_data = 2
All readers and writers have finished.


Program 6: Dining Philosophers Problem using Semaphores

In [6]:
import threading
import time

N = 5

# Semaphore for each fork
forks = [threading.Semaphore(1) for _ in range(N)]

# Allows at most N-1 philosophers to try to eat at once
room = threading.Semaphore(N - 1)


def philosopher(id):
    left = id
    right = (id + 1) % N

    for cycle in range(2):
        print(f"Philosopher {id} is thinking")
        time.sleep(0.1)

        # Enter the room
        room.acquire()

        # Pick up left and right forks
        forks[left].acquire()
        forks[right].acquire()

        print(
            f"Philosopher {id} is eating "
            f"(fork {left} and fork {right})"
        )

        time.sleep(0.15)

        # Put down the forks
        forks[right].release()
        forks[left].release()

        # Leave the room
        room.release()

        print(
            f"Philosopher {id} finished eating "
            "and put down forks"
        )


def main():
    philosophers = []

    # Create five philosopher threads
    for i in range(N):
        t = threading.Thread(target=philosopher, args=(i,))
        philosophers.append(t)
        t.start()

    # Wait for all philosophers to finish
    for t in philosophers:
        t.join()

    print("All philosophers have finished eating.")


if __name__ == "__main__":
    main()


Philosopher 0 is thinking
Philosopher 1 is thinking
Philosopher 2 is thinking
Philosopher 3 is thinking
Philosopher 4 is thinking
Philosopher 0 is eating (fork 0 and fork 1)
Philosopher 2 is eating (fork 2 and fork 3)
Philosopher 0 finished eating and put down forksPhilosopher 4 is eating (fork 4 and fork 0)

Philosopher 0 is thinking
Philosopher 2 finished eating and put down forks
Philosopher 2 is thinking
Philosopher 1 is eating (fork 1 and fork 2)
Philosopher 4 finished eating and put down forks
Philosopher 4 is thinking
Philosopher 3 is eating (fork 3 and fork 4)
Philosopher 1 finished eating and put down forks
Philosopher 1 is thinking
Philosopher 0 is eating (fork 0 and fork 1)
Philosopher 3 finished eating and put down forks
Philosopher 3 is thinking
Philosopher 0 finished eating and put down forks
Philosopher 2 is eating (fork 2 and fork 3)
Philosopher 4 is eating (fork 4 and fork 0)
Philosopher 2 finished eating and put down forks
Philosopher 4 finished eating and put down fo



Program 7: Deadlock Avoidance using Banker's Algorithm

In [9]:
P = 5  # Number of processes
R = 3  # Number of resource types
available = [0] * R
maximum = [[0] * R for _ in range(P)]
allocation = [[0] * R for _ in range(P)]
need = [[0] * R for _ in range(P)]

# Calculate Need Matrix
def calculateNeed():
    for i in range(P):
        for j in range(R):
            need[i][j] = maximum[i][j] - allocation[i][j]

# Check whether the system is in a safe state
def isSafe():
    work = available.copy()
    finish = [False] * P
    safeSeq = []
    count = 0

    while count < P:
        found = False

        for i in range(P):
            if not finish[i]:
                j = 0

                while j < R and need[i][j] <= work[j]:
                    j += 1

                if j == R:
                    for k in range(R):
                        work[k] += allocation[i][k]

                    safeSeq.append(i)
                    finish[i] = True
                    count += 1
                    found = True

        if not found:
            return False, []

    return True, safeSeq

# Request resources for a process
def requestResources(pid, request):

    if pid < 0 or pid >= P:
        print("Error: Invalid process number.")
        return False
    for j in range(R):

        if request[j] > need[pid][j]:
            print("Error: Process has exceeded its maximum claim.")
            return False

        if request[j] > available[j]:
            print("Request denied: Not enough resources available.")
            return False
    # Tentatively allocate resources
    for j in range(R):
        available[j] -= request[j]
        allocation[pid][j] += request[j]
        need[pid][j] -= request[j]

    safe, safeSeq = isSafe()

    if safe:
        print("Request can be granted safely.")
        print("Safe sequence:", " ".join(f"P{i}" for i in safeSeq))
        return True
    else:
        # Roll back allocation if the state is unsafe
        for j in range(R):
            available[j] += request[j]
            allocation[pid][j] -= request[j]
            need[pid][j] += request[j]

        print("Request denied: Would lead to an unsafe state.")
        return False
# Main program
def main():
    global available, maximum, allocation
    print(f"Enter available resources ({R} values):")
    available = list(map(int, input().split()))

    if len(available) != R or any(x < 0 for x in available):
        print("Invalid available resources.")
        return
    print(f"Enter Maximum matrix ({P} processes x {R} resources):")

    for i in range(P):
        maximum[i] = list(map(int, input(f"P{i}: ").split()))

        if len(maximum[i]) != R or any(x < 0 for x in maximum[i]):
            print("Invalid Maximum matrix.")
            return

    print("Enter Allocation matrix:")

    for i in range(P):
        allocation[i] = list(map(int, input(f"P{i}: ").split()))

        if len(allocation[i]) != R or any(x < 0 for x in allocation[i]):
            print("Invalid Allocation matrix.")
            return
    # Validate allocation against maximum claim
    for i in range(P):
        for j in range(R):
            if allocation[i][j] > maximum[i][j]:
                print("Error: Allocation exceeds maximum claim.")
                return
    # Calculate Need
    calculateNeed()

    print("\nNeed Matrix:")
    for i in range(P):
        print(f"P{i}: {need[i]}")

    # Check system safety
    safe, safeSeq = isSafe()
    if safe:
        print("\nSystem is in a SAFE state.")
        print("Safe sequence:", " ".join(f"P{i}" for i in safeSeq))
    else:
        print("\nSystem is in an UNSAFE state.")
    # Enter resource request
    try:
        pid = int(input("\nEnter process number requesting resources (0-4): "))
        request = list(map(int, input(f"Enter request vector ({R} values): ").split()))

        if len(request) != R or any(x < 0 for x in request):
            print("Invalid request vector.")
            return

        if not safe:
            print("Request denied: System is already in an unsafe state.")
            return
        requestResources(pid, request)
    except ValueError:
        print("Invalid input. Please enter integers.")
if __name__ == "__main__":
    main()


Enter available resources (3 values):
3 3 2
Enter Maximum matrix (5 processes x 3 resources):
P0: 7 5 3
P1: 3 2 2
P2: 9 0 2
P3: 2 2 2
P4: 4 3 3
Enter Allocation matrix:
P0: 0 1 0
P1: 2 0 0
P2: 3 0 2
P3: 2 1 1
P4: 0 2 2

Need Matrix:
P0: [7, 4, 3]
P1: [1, 2, 2]
P2: [6, 0, 0]
P3: [0, 1, 1]
P4: [4, 1, 1]

System is in a SAFE state.
Safe sequence: P1 P3 P4 P0 P2

Enter process number requesting resources (0-4): 1
Enter request vector (3 values): 1 0 2
Request can be granted safely.
Safe sequence: P1 P3 P4 P0 P2


Program 8: Contiguous Memory Allocation — First Fit, Best Fit and Worst Fit

In [11]:
MAX = 20


# First Fit Algorithm
def firstFit(bsize, psize):
    bn = len(bsize)
    pn = len(psize)

    block = bsize.copy()
    allocation = [-1] * pn

    for i in range(pn):
        for j in range(bn):
            if block[j] >= psize[i]:
                allocation[i] = j
                block[j] -= psize[i]
                break

    print("\n--- First Fit ---")
    print("Process No.\tProcess Size\tBlock No.")

    for i in range(pn):
        print(f"P{i + 1}\t\t{psize[i]}\t\t", end="")

        if allocation[i] != -1:
            print(f"B{allocation[i] + 1}")
        else:
            print("Not Allocated")


# Best Fit Algorithm
def bestFit(bsize, psize):
    bn = len(bsize)
    pn = len(psize)

    block = bsize.copy()
    allocation = [-1] * pn

    for i in range(pn):
        bestIdx = -1

        for j in range(bn):
            if block[j] >= psize[i]:
                if bestIdx == -1 or block[j] < block[bestIdx]:
                    bestIdx = j

        if bestIdx != -1:
            allocation[i] = bestIdx
            block[bestIdx] -= psize[i]

    print("\n--- Best Fit ---")
    print("Process No.\tProcess Size\tBlock No.")

    for i in range(pn):
        print(f"P{i + 1}\t\t{psize[i]}\t\t", end="")

        if allocation[i] != -1:
            print(f"B{allocation[i] + 1}")
        else:
            print("Not Allocated")


# Worst Fit Algorithm
def worstFit(bsize, psize):
    bn = len(bsize)
    pn = len(psize)

    block = bsize.copy()
    allocation = [-1] * pn

    for i in range(pn):
        worstIdx = -1

        for j in range(bn):
            if block[j] >= psize[i]:
                if worstIdx == -1 or block[j] > block[worstIdx]:
                    worstIdx = j

        if worstIdx != -1:
            allocation[i] = worstIdx
            block[worstIdx] -= psize[i]

    print("\n--- Worst Fit ---")
    print("Process No.\tProcess Size\tBlock No.")

    for i in range(pn):
        print(f"P{i + 1}\t\t{psize[i]}\t\t", end="")

        if allocation[i] != -1:
            print(f"B{allocation[i] + 1}")
        else:
            print("Not Allocated")


# Main Program
if __name__ == "__main__":

    bn = int(input("Enter number of memory blocks: "))

    if bn < 1 or bn > MAX:
        print("Number of blocks must be between 1 and 20.")
        raise SystemExit

    bsize = list(map(int, input("Enter sizes of blocks: ").split()))

    if len(bsize) != bn or any(x <= 0 for x in bsize):
        print("Enter valid positive sizes for all blocks.")
        raise SystemExit

    pn = int(input("Enter number of processes: "))

    if pn < 1 or pn > MAX:
        print("Number of processes must be between 1 and 20.")
        raise SystemExit

    psize = list(map(int, input("Enter sizes of processes: ").split()))

    if len(psize) != pn or any(x <= 0 for x in psize):
        print("Enter valid positive sizes for all processes.")
        raise SystemExit

    firstFit(bsize, psize)
    bestFit(bsize, psize)
    worstFit(bsize, psize)


Enter number of memory blocks: 5
Enter sizes of blocks: 110 520 200 340 620
Enter number of processes: 4
Enter sizes of processes: 212 412 115 426

--- First Fit ---
Process No.	Process Size	Block No.
P1		212		B2
P2		412		B5
P3		115		B2
P4		426		Not Allocated

--- Best Fit ---
Process No.	Process Size	Block No.
P1		212		B4
P2		412		B2
P3		115		B4
P4		426		B5

--- Worst Fit ---
Process No.	Process Size	Block No.
P1		212		B5
P2		412		B2
P3		115		B5
P4		426		Not Allocated


Program 9: Page Replacement Algorithms — FIFO, LRU and Optimal


In [12]:
MAXFRAMES = 10
MAXPAGES = 50


# Check whether a page is present in frames
def isPresent(frames, nf, page):
    for i in range(nf):
        if frames[i] == page:
            return True
    return False


# 1. FIFO Page Replacement
def fifo(ref, nf):
    frames = [-1] * nf
    front = 0
    faults = 0

    for page in ref:
        if not isPresent(frames, nf, page):
            frames[front] = page
            front = (front + 1) % nf
            faults += 1

    print("FIFO Page Faults  =", faults)


# 2. LRU Page Replacement
def lru(ref, nf):
    frames = [-1] * nf
    lastUsed = [0] * nf
    filled = 0
    faults = 0

    for i, page in enumerate(ref):
        pos = -1

        for j in range(nf):
            if frames[j] == page:
                pos = j
                break

        if pos == -1:
            faults += 1

            if filled < nf:
                frames[filled] = page
                lastUsed[filled] = i
                filled += 1

            else:
                # Find the least recently used page
                lruIdx = 0

                for j in range(1, nf):
                    if lastUsed[j] < lastUsed[lruIdx]:
                        lruIdx = j

                frames[lruIdx] = page
                lastUsed[lruIdx] = i

        else:
            lastUsed[pos] = i

    print("LRU Page Faults   =", faults)


# 3. Optimal Page Replacement
def optimal(ref, nf):
    frames = [-1] * nf
    filled = 0
    faults = 0

    for i, page in enumerate(ref):

        if isPresent(frames, filled, page):
            continue

        faults += 1

        if filled < nf:
            frames[filled] = page
            filled += 1
            continue

        # Find the page used farthest in the future
        replaceIdx = -1
        farthest = -1

        for j in range(nf):
            try:
                k = ref.index(frames[j], i + 1)
            except ValueError:
                # This page will never be used again
                replaceIdx = j
                break

            if k > farthest:
                farthest = k
                replaceIdx = j

        frames[replaceIdx] = page

    print("Optimal Page Faults =", faults)


# Main Program
if __name__ == "__main__":

    n = int(input("Enter number of pages in reference string: "))

    if n < 1 or n > MAXPAGES:
        print("Number of pages must be between 1 and 50.")
        raise SystemExit

    ref = list(map(int, input("Enter the reference string: ").split()))

    if len(ref) != n or any(page < 0 for page in ref):
        print("Enter exactly", n, "valid non-negative page numbers.")
        raise SystemExit

    nf = int(input("Enter number of frames: "))

    if nf < 1 or nf > MAXFRAMES:
        print("Number of frames must be between 1 and 10.")
        raise SystemExit

    fifo(ref, nf)
    lru(ref, nf)
    optimal(ref, nf)





Enter number of pages in reference string: 12
Enter the reference string: 7 0 1 2 0 3 0 4 2 3 0 3
Enter number of frames: 3
FIFO Page Faults  = 10
LRU Page Faults   = 9
Optimal Page Faults = 7


Program 10: Disk Scheduling Algorithms — FCFS, SSTF, SCAN and C-SCAN




In [13]:
MAX = 50


def abs_diff(a, b):
    return abs(a - b)


def fcfs(req, n, head):
    total = 0
    cur = head

    for i in range(n):
        total += abs_diff(cur, req[i])
        cur = req[i]

    print("FCFS   total head movement =", total)


def sstf(req, n, head):
    visited = [0] * n
    total = 0
    cur = head

    for c in range(n):
        idx = -1
        best = float('inf')

        for i in range(n):
            if not visited[i] and abs_diff(cur, req[i]) < best:
                best = abs_diff(cur, req[i])
                idx = i

        visited[idx] = 1
        total += best
        cur = req[idx]

    print("SSTF   total head movement =", total)


def scan(req, n, head, diskSize):
    arr = req.copy()
    arr.append(head)
    arr.sort()

    idx = arr.index(head)

    total = 0
    cur = head

    # Move towards the higher end first
    for i in range(idx + 1, len(arr)):
        total += abs_diff(cur, arr[i])
        cur = arr[i]

    # Move to the last cylinder
    total += abs_diff(cur, diskSize - 1)
    cur = diskSize - 1

    # Reverse direction
    for i in range(idx - 1, -1, -1):
        total += abs_diff(cur, arr[i])
        cur = arr[i]

    print("SCAN   total head movement =", total)


def cscan(req, n, head, diskSize):
    arr = req.copy()
    arr.append(head)
    arr.sort()

    idx = arr.index(head)

    total = 0
    cur = head

    # Move towards the higher end first
    for i in range(idx + 1, len(arr)):
        total += abs_diff(cur, arr[i])
        cur = arr[i]

    # Move to the last cylinder
    total += abs_diff(cur, diskSize - 1)

    # Jump from the last cylinder to cylinder 0
    total += diskSize - 1
    cur = 0

    # Continue moving towards higher cylinders
    for i in range(idx):
        total += abs_diff(cur, arr[i])
        cur = arr[i]

    print("C-SCAN total head movement =", total)


if __name__ == "__main__":
    n = int(input("Enter number of requests: "))

    if n < 1 or n > MAX:
        print("Number of requests must be between 1 and 50.")
        raise SystemExit

    req = list(map(int, input("Enter the requests: ").split()))

    if len(req) != n:
        print("Please enter exactly", n, "requests.")
        raise SystemExit

    head = int(input("Enter initial head position: "))
    diskSize = int(input("Enter disk size (max cylinder + 1): "))

    if diskSize < 1 or not (0 <= head < diskSize):
        print("Invalid disk size or initial head position.")
        raise SystemExit

    if any(r < 0 or r >= diskSize for r in req):
        print("All requests must be between 0 and", diskSize - 1)
        raise SystemExit

    fcfs(req, n, head)
    sstf(req, n, head)
    scan(req, n, head, diskSize)
    cscan(req, n, head, diskSize)



Enter number of requests: 8
Enter the requests: 90 180 40 120 14 114 65 67
Enter initial head position: 55
Enter disk size (max cylinder + 1): 200
FCFS   total head movement = 602
SSTF   total head movement = 291
SCAN   total head movement = 329
C-SCAN total head movement = 383


Program 11: File Allocation Methods — Contiguous, Linked and Indexed Allocation

In [14]:

MAXBLOCKS = 30
MAXFILES = 10

disk = [0] * MAXBLOCKS  # 0 = free, 1 = allocated


def resetDisk():
    global disk
    disk = [0] * MAXBLOCKS


def contiguousAllocation():
    resetDisk()

    print("\n--- Contiguous Allocation ---")
    nf = int(input("Enter number of files: "))

    for f in range(nf):
        start, length = map(
            int, input(f"File {f + 1} - starting block, length: ").split()
        )

        ok = True

        if start < 0 or length <= 0 or start + length > MAXBLOCKS:
            ok = False
        else:
            for i in range(start, start + length):
                if disk[i] == 1:
                    ok = False
                    break

        if not ok:
            print("Allocation failed: blocks not free/out of range.")
            continue

        for i in range(start, start + length):
            disk[i] = 1

        print(
            f"File {f + 1} allocated blocks "
            f"{start} to {start + length - 1}"
        )


def linkedAllocation():
    resetDisk()

    print("\n--- Linked Allocation ---")
    nf = int(input("Enter number of files: "))

    for f in range(nf):
        length = int(input(f"File {f + 1} - number of blocks: "))

        if length <= 0:
            print("Invalid number of blocks.")
            continue

        blocks = []

        for i in range(MAXBLOCKS):
            if disk[i] == 0 and len(blocks) < length:
                disk[i] = 1
                blocks.append(i)

        print(f"File {f + 1} allocated blocks: ", end="")

        if blocks:
            print(" -> ".join(map(str, blocks)), end=" -> ")

        print("(NULL)")

        if len(blocks) < length:
            print("Warning: disk full, could not allocate all blocks.")


def indexedAllocation():
    resetDisk()

    print("\n--- Indexed Allocation ---")
    nf = int(input("Enter number of files: "))

    for f in range(nf):
        length = int(
            input(f"File {f + 1} - number of data blocks needed: ")
        )

        if length < 0:
            print("Invalid number of blocks.")
            continue

        indexBlock = -1

        for i in range(MAXBLOCKS):
            if disk[i] == 0:
                indexBlock = i
                disk[i] = 1
                break

        if indexBlock == -1:
            print(f"File {f + 1}: Allocation failed. No index block available.")
            continue

        dataBlocks = []

        for i in range(MAXBLOCKS):
            if disk[i] == 0 and len(dataBlocks) < length:
                disk[i] = 1
                dataBlocks.append(i)

        print(
            f"File {f + 1} - Index block: {indexBlock}, "
            f"Data blocks: ", end=""
        )

        print(*dataBlocks)

        if len(dataBlocks) < length:
            print("Warning: disk full, could not allocate all data blocks.")


if __name__ == "__main__":
    print("Menu:")
    print("1. Contiguous Allocation")
    print("2. Linked Allocation")
    print("3. Indexed Allocation")

    choice = int(input("Enter choice: "))

    if choice == 1:
        contiguousAllocation()
    elif choice == 2:
        linkedAllocation()
    elif choice == 3:
        indexedAllocation()
    else:
        print("Invalid choice")


Menu:
1. Contiguous Allocation
2. Linked Allocation
3. Indexed Allocation
Enter choice: 2

--- Linked Allocation ---
Enter number of files: 2
File 1 - number of blocks: 3
File 1 allocated blocks: 0 -> 1 -> 2 -> (NULL)
File 2 - number of blocks: 4
File 2 allocated blocks: 3 -> 4 -> 5 -> 6 -> (NULL)
